# DeterminAgent: screen small models on a free GPU (Phase 0.4)

This notebook runs the 0.4 screen on a free T4 GPU in Google Colab or Kaggle. It installs Ollama, pulls each model, runs the golden pilot cases through the eval runner, scores them and saves the results to download. No API key is needed and nothing is paid for.

**Steps**
1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU T4**, and Settings → **Internet on** (Kaggle needs a verified phone number for both).
2. Edit the settings cell if you want fewer models. The default set takes about 30–60 minutes.
3. Runtime → **Run all**.
4. At the end, download `determinagent-screen.zip` (Colab downloads it, Kaggle puts it under Output) and send it back.

Free sessions stop after a few hours or when idle. Results are written after each model, so a stopped session keeps what finished.

In [ ]:
# Settings
BRANCH = "slm-strategy"  # repo branch with the runner; use "main" once PR #6 is merged
GOLDEN = "golden/v1/pilot.yaml"
TASKS = ["rulespeak_validation", "rulespeak_generation", "gherkin_requirements"]
LIMIT = None  # e.g. 5 for a quick smoke test of each task

# Ollama tags, smallest first. All fit a 16 GB T4 at Ollama's default 4-bit quantization.
MODELS = [
    "qwen3:0.6b",
    "smollm2:360m",
    "gemma3:270m",
    "qwen3:1.7b",
    "qwen3:4b",
    "phi4-mini",
    "qwen3:8b",
]
# More shortlist models. Check each tag on https://ollama.com/library first, then add it above:
#   gemma4 (E4B, 12B), ministral-3 (3B, 8B), granite4.1 (3B, 8B), smollm3
REMOVE_AFTER_RUN = True  # delete each model after its run, to save disk

In [ ]:
import os
import pathlib
import subprocess

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content")
print("Kaggle" if IN_KAGGLE else "Colab (or other)", "| work dir:", WORK)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True
    ).stdout
    or "No GPU found: models will run on CPU, slowly."
)

## 1. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1 && echo "ollama installed" || tail -20 /tmp/ollama-install.log

In [ ]:
import time
import urllib.request

log = open("/tmp/ollama.log", "w")
server = subprocess.Popen(
    ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_KEEP_ALIVE": "30m"}
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is up:", urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /tmp/ollama.log")

## 2. Get the code

In [ ]:
REPO = WORK / "determinagent"
if not REPO.exists():
    !git clone -q --branch {BRANCH} https://github.com/joel-waddell/determinagent.git {REPO}
!pip install -q -e {REPO}
os.chdir(REPO)
!git log --oneline -1

## 3. Run the screen

Each model runs every selected case with prompt v1, two checker-feedback retries and schema-constrained output. One JSONL run file per model goes to `runs/`, which `python -m determinagent.eval score` and `compare` read.

In [ ]:
import json

import yaml

from determinagent.eval import score_run
from determinagent.eval.runner import run
from determinagent.models import from_spec

cases_all = yaml.safe_load(open(GOLDEN))["cases"]
cases = [c for c in cases_all if c["task"] in TASKS]
if LIMIT:
    cases = [c for t in TASKS for c in [x for x in cases if x["task"] == t][:LIMIT]]
RUNS = REPO / "runs"
RUNS.mkdir(exist_ok=True)
summary = []


def summarize(model, recs):
    rows = []
    rep = score_run(cases, recs, system=f"ollama:{model}")
    for t in TASKS:
        m = rep.subset(task=t).metrics()
        if not m["cases"]:
            continue
        rows.append(
            {
                "model": model,
                "task": t,
                "cases": m["cases"],
                "task_success": m["task_success"],
                "first_pass": m["wellformed_first_pass"],
                "wellformed": m["wellformed_final"],
                "violations": m["violation_rate"],
                "precision": m.get("validation_precision"),
                "recall": m.get("validation_recall"),
                "p95_ms": m["latency_p95_ms"],
            }
        )
    return rows


for model in MODELS:
    out = RUNS / f"{model.replace(':', '_').replace('/', '_')}.jsonl"
    if out.exists():
        print(f"{model}: already run, skipping")
        summary += summarize(model, [json.loads(line) for line in open(out)])
        continue
    print(f"\n=== {model}: pulling")
    if subprocess.run(["ollama", "pull", model], capture_output=True, text=True).returncode != 0:
        print(f"{model}: pull failed (check the tag), skipping")
        continue
    t0 = time.time()
    recs = run(cases, from_spec(f"ollama:{model}"), retries=2)
    out.write_text("".join(json.dumps(r) + "\n" for r in recs))
    rows = summarize(model, recs)
    summary += rows
    print(f"{model}: {len(recs)} cases in {time.time() - t0:.0f}s")
    for r in rows:
        print(
            f"  {r['task']:<22} success {r['task_success']:.0%}  first-pass {r['first_pass']:.0%}  p95 {r['p95_ms'] or 0:.0f} ms"
        )
    if REMOVE_AFTER_RUN:
        subprocess.run(["ollama", "rm", model], capture_output=True)

## 4. Results

In [ ]:
import pandas as pd

df = pd.DataFrame(summary)
(RUNS / "summary.csv").write_text(df.to_csv(index=False))
pd.set_option("display.float_format", "{:.2f}".format)
df.pivot_table(index="model", columns="task", values="task_success", sort=False) if len(df) else df

In [ ]:
df

## 5. Download

This zips `runs/` (one run file per model plus `summary.csv`). Send the zip back to have it scored and written up.

In [ ]:
import shutil

zip_path = shutil.make_archive(str(WORK / "determinagent-screen"), "zip", RUNS)
print("Saved", zip_path)
if not IN_KAGGLE:
    try:
        from google.colab import files

        files.download(zip_path)
    except ImportError:
        pass
else:
    print("On Kaggle: open the Output panel (right side) to download it.")

## Optional: expose this Ollama server

Skip this section for the normal screen. It is for running the screen from somewhere else against this GPU. A Cloudflare quick tunnel gives a public URL with **no authentication**, so anyone with the URL can use the server while the notebook runs. Stop the cell when you're done.

From the other machine:
`OLLAMA_BASE_URL=<url>/v1 python -m determinagent.eval run golden/v1/pilot.yaml --model ollama:qwen3:0.6b --out runs/qwen3.jsonl`

In [ ]:
EXPOSE = False  # set True to open a tunnel
if EXPOSE:
    !curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
    import re

    tun = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in tun.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("Ollama endpoint:", m.group(0) + "/v1")
            break